# Stage 1: Data Audit Notebook
## Amazon ML Challenge — Business Entity Resolution

This notebook inspects the raw dataset files (`train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, `train_ground_truth.tsv`, `test_source1.tsv`, etc.), computes row counts, missing values, duplicates, country distributions, and ground-truth match statistics.

In [ ]:
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.stage_runner import StageController
from src.utils.storage import StorageManager
from src.data.audit import run_data_audit

### Step 1: Execute Data Audit Stage

In [ ]:
storage = StorageManager("../configs/config.yaml")
storage.initialize_directories()

# Run audit
controller = StageController("../configs/config.yaml")
success = controller.run_stage("data_audit", force=True)

### Step 2: Display Structured Audit Report

In [ ]:
report_path = storage.artifacts_dir / "audit_report.json"
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        audit_report = json.load(f)
    
    print("=" * 80)
    print("                 STAGE 1: DATA AUDIT SUMMARY REPORT                 ")
    print("=" * 80)
    for tbl, info in audit_report.get("tables", {}).items():
        if "num_rows" in info:
            print(f"\nTable: {tbl}")
            print(f"  - Rows: {info['num_rows']:,}, Cols: {info['num_cols']}")
            print(f"  - Unique Entity IDs: {info.get('unique_entity_ids', 'N/A')}")
            print(f"  - Exact Duplicate Rows: {info.get('total_exact_duplicate_rows', 0)}")
            if 'country_distribution' in info:
                print(f"  - Country Distribution: {info['country_distribution']}")
    
    gt = audit_report.get("ground_truth_audit", {})
    if gt and 'total_source1_entities' in gt:
        print("\n" + "-" * 80)
        print("GROUND TRUTH & MATCH CARDINALITY:")
        print(f"  - Total Source 1 Entities : {gt['total_source1_entities']:,}")
        print(f"  - Singletons (0 matches)  : {gt['singletons_count']:,} ({gt['singletons_pct']}%)")
        print(f"  - One-to-One Matches      : {gt['one_to_one_count']:,} ({gt['one_to_one_pct']}%)")
        print(f"  - One-to-Many Matches     : {gt['one_to_many_count']:,} ({gt['one_to_many_pct']}%)")
        print(f"  - Max Matches per Entity  : {gt['max_matches_per_s1']}")
        print(f"  - Mean Matches per Entity : {gt['mean_matches_per_s1']}")
        print(f"  - S2 Targets / S3 Targets : {gt['matched_targets_source2']:,} / {gt['matched_targets_source3']:,}")
        print("=" * 80)
else:
    print("Audit report not found. Please ensure data is downloaded to data/raw/ and run Step 1.")